# Get academic article's data from Open Alex.

## Preparation and query

In [ ]:
# openalex credentials

# No credentials are needed.

In [ ]:
# Load libraries
import requests
import pandas as pd
import time
import random
import os
import pickle

In [ ]:
# mount my google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Folder to save the results
DATAFOLDER = "drive/MyDrive/SynBio/20251104"

In [ ]:
# # Query. Term search
# KEYWORDS = '''
#   ("energy storage" OR "energy harvesting" OR "supercapacitor" OR "pseudocapacitor")
# '''
# #patent* AND ("artificial intelligence" OR "machine learning")
# #@markdown The start/end year of publications used to extract patents
# YEAR_START = 2015 #@param {type: "slider", min: 1950, max: 2023}
# YEAR_END = 2025  #@param {type: "slider", min: 1950, max: 2023}

# if YEAR_END < YEAR_START:
#   YEAR_END = YEAR_START

In [ ]:
# # Journals: First find the id of the journal
# # https://docs.openalex.org/api-entities/sources/filter-sources
# journal_endpoint = "https://api.openalex.org/sources"
# journal_q = "ieee"
# response = requests.get(f'{journal_endpoint}?search={journal_q}&select=id,issn,display_name,works_count,cited_by_count,alternate_titles')
# #response.json()
# pd.json_normalize(response.json()['results']).head()

In [ ]:
# test = pd.json_normalize(response.json()['results'])
# test = test['id'].apply(lambda x: x.replace('https://openalex.org/', ''))
# ieee_ids = list(set(test.to_list()))
# ieee_ids

In [ ]:
# Query. Advanced search

# # comma separated list of ISO 2-digit codes
# countries = ["JP","US"]

# # comma separated list of openalex ids
# institutions = []
# authors = []
# journals = ieee_ids

# # Other filters allowed by openalex
# # numeric filters are = > <. Not possible with ">=" or "=<"
#advanced = 'cited_by_count:>100'



---



In [ ]:
# Create the folder if doesn't exists
if not os.path.exists(DATAFOLDER):
  !mkdir $DATAFOLDER
  print(f"==\nCreated data folder:", DATAFOLDER + "/")



---



## Utils
### Functions for querying and downloading

In [ ]:
# New format query 20250416

# Format term search query to openalex syntax
def format_query_openalex(search_terms):
  return f'title_and_abstract.search:{search_terms}'

In [ ]:
# # Format term search query to openalex syntax
# def format_query_openalex(search_terms):
#   '''
#   Takes a query string containing AND and OR operators and converts it to the openalex syntax.
#   AND and OR must be uppercased and preceeded and succeded by a space.
#   Parenthesys are allowed. NESTED parenthesis are NOT allowed. Asterisks will be removed. Quotation marks are allowed.
#   This is a simply parser, thus it only handles the case when the AND operators are at the first level.
#   We cannot have AND inside an OR parenthesis. e.g. `patent* OR (property AND rights)` will fail.
#   `patent* AND ("artificial intelligence" OR "machine learning")` will work.
#   '''
#   q = search_terms.replace('*', '').strip()
#   q = q.split(" AND ")
#   formatted_q = []
#   for part in q:
#     tmp = part.replace(" OR ", "|").replace(")","").replace("(","").replace("'","").replace('"','').strip()
#     tmp = tmp.replace(' ', '%20')
#     #formatted_q.append(f'title.search:{tmp}')   # <-- ⚠️It must be either title OR abstract. If we uncomment both openalex will apply AND automatically. Meaning the keywords match title AND abstract which will reduce the size of the dataset
#     formatted_q.append(f'abstract.search:{tmp}')
#   formatted_q = ','.join(formatted_q)
#   return formatted_q

In [ ]:
# create query string
def create_query_string_for_openalex(keywords, from_year, to_year, countries=[], authors=[], institutions=[], journals=[], advanced=''):
  '''
  Creates the query string used to call openalex.
  Printing this string and put it in a browser should work.
  '''
  main_string = []

  # Works endpoint
  works_endpoint = "https://api.openalex.org/works"

  # Add user input
  if keywords.strip() != '':
    main_string.append(format_query_openalex(keywords))
  if from_year != '':
    main_string.append(f'from_publication_date:{from_year}-01-01')
  if to_year != '':
    main_string.append(f'to_publication_date:{to_year}-12-31')
  if type(countries) == type([]) and countries != []:
    main_string.append(f"institutions.country_code:{'|'.join(countries)}")
  if type(authors) == type([]) and authors != []:
    main_string.append(f"authorships.author.id:{'|'.join(authors)}")
  if type(institutions) == type([]) and institutions != []:
    main_string.append(f"institutions.id:{'|'.join(institutions)}")
  if type(journals) == type([]) and journals != []:
    main_string.append(f"locations.source.id:{'|'.join(journals)}")
  if type(advanced) == type('') and advanced != '':
    advanced = advanced.replace('\n','').strip()
    main_string.append(advanced)

  # Add hardcoded filters
  paper_type = 'type:article'
  #language = 'language:en'
  #retracted = 'is_retracted:false'

  main_string.append(paper_type)
  #main_string.append(language)
  #main_string.append(retracted)
  main_string = ','.join(main_string)

  # Other query parameters
  # Sorting
  sorting = 'cited_by_count:desc'

  # Select
  selection = 'id,doi,title,publication_year,language,type,primary_location,open_access,authorships,cited_by_count,referenced_works,abstract_inverted_index,concepts,is_retracted'

  # Authentication
  email = f"mailto={random.choice(('cristian@jiyu-labs.com','admin@jiyu-labs.com'))}"

  # Full query
  full_query = f'{works_endpoint}?filter={main_string}&sort={sorting}&select={selection}&{email}'

  return full_query

In [ ]:
def download_from_openalex(formatted_query, records_per_call = 200, pause_time = 1):
  '''
  Iteratively download the records from Open Alex.
  formatted_query: is the output from `create_query_string_for_openalex()`
  records_per_call: the number of papers to download per API call. The max allowed is 200. If the server report errors or becomes slow use a lower value
  pause_time: Open Alex is a public service, to not overload their server we pasue between calls.
  '''
  # Pagination
  per_page = records_per_call #The max is 200.
  cursor = '*' #The first page is always '*' in cursor pagination

  # Loop for downloading the records.
  papers = []
  while cursor is not None:
    loop_query = f'{formatted_query}&per-page={per_page}&cursor={cursor}'
    response = requests.get(loop_query)
    print(response.json()['meta'])
    papers = papers + response.json()['results']
    cursor = response.json()['meta']['next_cursor']
    time.sleep(max(0.5, pause_time))

  return papers

## Change OpenAlex format to Dimensions format for compatibility

In [ ]:
def format_abstract(abstract_inverted_index):
  ''' Takes an abstract inverted index from OpenAlex and transforms it to regular text'''
  word_index = []
  if (abstract_inverted_index is not None) and (len(abstract_inverted_index) > 0):
    for k,v in abstract_inverted_index.items():
      for index in v:
        word_index.append([k,index])
    word_index = sorted(word_index,key = lambda x : x[1])
  return(' '.join([i[0] for i in word_index]))

In [ ]:
def format_concepts(concepts):
  '''takes the concept object from Open Alex and transform it to a list as in Dimensions.
     Open Alex concepts have a score. We take those with a score >=0.3'''
  formatted_concepts = []
  if len(concepts) > 0:
    formatted_concepts = [concept['display_name'] for concept in concepts if concept['score'] >= 0.3]
  return formatted_concepts

In [ ]:
def format_journal(primary_location, doi, infer = False):
  '''takes the primary location dictionary and format it in the JSON expected by dimensions'''
  formatted_journal = {'id': '', 'title': ''}
  if (primary_location is not None) and (isinstance(primary_location, dict)) and ('source' in primary_location.keys()):
    if primary_location['source'] is not None:
      #print('Formatting journal name with provided openalex data')
      formatted_journal = {
                            'id': primary_location['source']['id'],
                            'title': primary_location['source']['display_name']
                          }
    else:
      if (infer) and (doi is not None):
        print(f'Finding journal name from Crossref for doi: {doi}')
        #print(primary_location['source'])
        try:
          crossref_response = requests.get(f'https://api.crossref.org/works?filter=doi:{doi.replace("https://doi.org/","")}&select=short-container-title&mailto=cristianmejia00@gmail.com')
          journal_name = crossref_response.json()['message']['items'][0]['short-container-title'][0]
        except Exception as e:
          print(e)
          journal_name = ''
        finally:
          formatted_journal = {
                        'id': journal_name,
                        'title': journal_name
          }
  return formatted_journal

In [ ]:
def find_if_open_access(primary_location):
  is_oa = {}
  if (primary_location is not None) and (isinstance(primary_location, dict)) and ('is_oa' in primary_location.keys()):
    is_oa = {
        'is_oa': primary_location['is_oa'],
        'oa_pdf': primary_location['pdf_url']
    }
  return is_oa

In [ ]:
country_codes = pd.read_csv("https://raw.githubusercontent.com/cristianmejia00/display/refs/heads/main/ISO3166_country_codes.csv")
iso_country = {}
for i in range(0,len(country_codes)):
  iso_country[country_codes['alpha_2_code'][i]] = country_codes['country'][i]
country_codes.head()

,country,alpha_2_code,alpha_3_code,numeric_code
0,Afghanistan,AF,AFG,4
1,Albania,AL,ALB,8
2,Algeria,DZ,DZA,12
3,American Samoa,AS,ASM,16
4,Andorra,AD,AND,20


In [ ]:
def format_authorship(authorships):
  '''
  Takes the authorships object from openalex and extracts the authors, institutions, and countries in the format expected by dimensions.
  '''
  institutions = []
  countries = []
  authors = []
  if (authorships is not None) and (len(authorships) > 0):
    for i in authorships:
      try:
        authors.append({'last_name': i['author']['display_name']})
      except KeyError as e:
        print(e)
      for j in i['institutions']:
        try:
          institutions.append(j['display_name'])
        except KeyError as e:
          print(e)
        try:
          if j['country_code'] is not None:
            countries.append(iso_country[j['country_code'].upper()])
        except KeyError as e:
          print(e)
  else:
    authors = []
  institutions = list(set(institutions))
  countries = list(set(countries))
  return authors, institutions, countries

In [ ]:
def sanitize_openalex_json(paper):
  '''
  Verify that every downloaded record has all necessary fields.
  When missing, we add properly formatted empty strings.
  This function allows avoinding the `KeyError` further down the road.
  '''
  # Every oap must have these keys
  if 'authorships' not in paper.keys():
    paper['authorships'] = []
  if 'id' not in paper.keys():
    paper['id'] = f'missing_oaid_{random.randint(100_000,999_999)}'
  if 'doi' not in paper.keys():
    paper['doi'] = ''
  if 'type' not in paper.keys():
    paper['type'] = 'unknown'
  if 'title' not in paper.keys():
    paper['title'] = 'unknown'
  if 'publication_year' not in paper.keys():
    paper['publication_year'] = 0
  if 'cited_by_count' not in paper.keys():
      paper['cited_by_count'] = 0
  if 'abstract_inverted_index' not in paper.keys():
      paper['abstract_inverted_index'] = []
  if 'concepts' not in paper.keys():
      paper['concepts'] = []
  if 'primary_location' not in paper.keys():
      print(f"this paper does not have a primary location: {paper['id']}")
      paper['primary_location'] = {}

  return paper

In [ ]:
def format_from_openalex_to_dimensions(papers):
  '''
  Format the list of papers downloaded with the Openalex API using `download_from_openalex()`
  to have the format of dimensions. So that we can use the rest of zenronbun.
  '''
  # Use Crossref for small datasets only. Too slow for large datasets.
  if len(papers) < 10000:
    infer = False # 🔴 change to True when needed!
  else:
    infer = False

  # Init
  papers_dimensions = []

  for oap in papers:
    try:
      oap = sanitize_openalex_json(oap)
      authors, institutions, countries = format_authorship(oap['authorships'])
      dimensions_json = {
          'id':                         oap['id'],
          'doi':                        oap['doi'],
          'type':                       oap['type'].replace('journal-', ''),
          'title':                      oap['title'],
          'year':                       int(oap['publication_year']),
          'times_cited':                int(oap['cited_by_count']),
          'abstract':                   format_abstract(oap['abstract_inverted_index']),
          'concepts':                   format_concepts(oap['concepts']),
          'journal.title':              format_journal(oap['primary_location'], oap['doi'], infer = infer),
          'researchers':                authors,
          'research_org_names':         institutions,
          'research_org_country_names': countries,
          'reference_ids':              oap['referenced_works'],
          'is_oa':                      find_if_open_access(oap['primary_location']),
          'is_retracted':                oap['is_retracted']
      }
    except Exception as e:
      print(e)
      print(oap['doi'])
      print(f"https://api.openalex.org/works/{oap['doi']}")
      raise
    papers_dimensions.append(dimensions_json)
  return papers_dimensions



---



## 🔴 Download data

In [ ]:
# NEW 20250416

# Query. Term search
KEYWORDS = '''
("artificial nucleic acid" OR "artificial nucleic acids" OR "artificial oligonucleotide" OR "artificial oligonucleotides" OR "artificial nucleotide" OR"artificial dinucleotide" OR "artificial internucleotide" OR "artificial oligodeoxynucleotide" OR "artificial polyribonucleotide" OR "bio brick" OR "biobrick" OR "bio-brick")
'''
#patent* AND ("artificial intelligence" OR "machine learning")
#@markdown The start/end year of publications used to extract patents
YEAR_START = 1950 #@param {type: "slider", min: 1950, max: 2025}
YEAR_END = 2025  #@param {type: "slider", min: 1950, max: 2025}

if YEAR_END < YEAR_START:
  YEAR_END = YEAR_START


############## DO NOT CHANGE!
# Check the publications available
formatted_query = create_query_string_for_openalex(KEYWORDS, YEAR_START, YEAR_END) #advanced=advanced
print('Click this link to verify the JSON data from openalex')
print(formatted_query)
print(' ')
response = requests.get(formatted_query)
print(f"Total articles for this query: {response.json()['meta']['count']}")

Click this link to verify the JSON data from openalex
https://api.openalex.org/works?filter=title_and_abstract.search:
("artificial nucleic acid" OR "artificial nucleic acids" OR "artificial oligonucleotide" OR "artificial oligonucleotides" OR "artificial nucleotide" OR"artificial dinucleotide" OR "artificial internucleotide" OR "artificial oligodeoxynucleotide" OR "artificial polyribonucleotide" OR "bio brick" OR "biobrick" OR "bio-brick")
,from_publication_date:1950-01-01,to_publication_date:2025-12-31,type:article&sort=cited_by_count:desc&select=id,doi,title,publication_year,language,type,primary_location,open_access,authorships,cited_by_count,referenced_works,abstract_inverted_index,concepts,is_retracted&mailto=cristian@jiyu-labs.com
 
Total articles for this query: 643


In [ ]:
test = response.json()['results'][0]
test

{'id': 'https://openalex.org/W2113983551',
 'doi': 'https://doi.org/10.1186/1754-1611-2-5',
 'title': 'Engineering BioBrick vectors from BioBrick parts',
 'publication_year': 2008,
 'language': 'en',
 'type': 'article',
 'primary_location': {'id': 'doi:10.1186/1754-1611-2-5',
  'is_oa': True,
  'landing_page_url': 'https://doi.org/10.1186/1754-1611-2-5',
  'pdf_url': 'https://jbioleng.biomedcentral.com/track/pdf/10.1186/1754-1611-2-5',
  'source': {'id': 'https://openalex.org/S156566788',
   'display_name': 'Journal of Biological Engineering',
   'issn_l': '1754-1611',
   'issn': ['1754-1611'],
   'is_oa': True,
   'is_in_doaj': True,
   'is_core': True,
   'host_organization': 'https://openalex.org/P4310320256',
   'host_organization_name': 'BioMed Central',
   'host_organization_lineage': ['https://openalex.org/P4310320256'],
   'host_organization_lineage_names': [],
   'type': 'journal'},
  'license': 'cc-by',
  'license_id': 'https://openalex.org/licenses/cc-by',
  'version': 'publ

In [ ]:
# Download the data
papers = download_from_openalex(formatted_query)
len(papers)

{'count': 643, 'db_response_time_ms': 35, 'page': None, 'per_page': 200, 'next_cursor': 'IlsxMiwgOTguMCwgMTIsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMzIwODkyNjE0NyddIg==', 'groups_count': None}
{'count': 643, 'db_response_time_ms': 43, 'page': None, 'per_page': 200, 'next_cursor': 'IlsyLCAnLUluZmluaXR5JywgMiwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1cyMDYzMzIyOTIyJ10i', 'groups_count': None}
{'count': 643, 'db_response_time_ms': 34, 'page': None, 'per_page': 200, 'next_cursor': 'IlswLCAnLUluZmluaXR5JywgMCwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1c0MjQ0MjQwNTY3J10i', 'groups_count': None}
{'count': 643, 'db_response_time_ms': 29, 'page': None, 'per_page': 200, 'next_cursor': 'IlswLCAnLUluZmluaXR5JywgMCwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1c4MDY0NDQ3MTAnXSI=', 'groups_count': None}
{'count': 643, 'db_response_time_ms': 24, 'page': None, 'per_page': 200, 'next_cursor': None, 'groups_count': None}


643

In [ ]:
# Convert to dimensions format
papers_dimensions = format_from_openalex_to_dimensions(papers)

In [ ]:
# Convert records to dataframe
pubs = pd.DataFrame().from_dict(papers_dimensions)
print("Publications: ", len(pubs))
pubs.drop_duplicates(subset='id', inplace=True)
print("Unique Publications: ", len(pubs))

pubs.head(5)

Publications:  643
Unique Publications:  643


,id,doi,type,title,year,times_cited,abstract,concepts,journal.title,researchers,research_org_names,research_org_country_names,reference_ids,is_oa,is_retracted
0,https://openalex.org/W2113983551,https://doi.org/10.1186/1754-1611-2-5,article,Engineering BioBrick vectors from BioBrick parts,2008,750,,"[Standardization, Reuse, Computer science, Pro...","{'id': 'https://openalex.org/S156566788', 'tit...","[{'last_name': 'Reshma Shetty'}, {'last_name':...","[None, IIT@MIT]",[USA],"[https://openalex.org/W2115286109, https://ope...","{'is_oa': True, 'oa_pdf': 'https://jbioleng.bi...",False
1,https://openalex.org/W2020246057,https://doi.org/10.1126/science.1080587,article,A Discrete Self-Assembled Metal Array in Artif...,2003,558,DNA has a structural basis to array functional...,"[Nucleobase, Unpaired electron, Copper, Crysta...","{'id': 'https://openalex.org/S3880285', 'title...","[{'last_name': 'Kentaro Tanaka'}, {'last_name'...","[The University of Tokyo, Institute for Molecu...",[Japan],"[https://openalex.org/W4234876890, https://ope...","{'is_oa': False, 'oa_pdf': None}",False
2,https://openalex.org/W1982228455,https://doi.org/10.1186/1754-1611-3-4,article,Measuring the activity of BioBrick promoters u...,2009,430,Abstract Background The engineering of many-co...,"[Promoter, Computer science, Computational bio...","{'id': 'https://openalex.org/S4306400425', 'ti...","[{'last_name': 'Jason Kelly'}, {'last_name': '...","[Virginia Tech, Johns Hopkins University, Stan...","[United Kingdom, USA]","[https://openalex.org/W206917024, https://open...","{'is_oa': True, 'oa_pdf': None}",False
3,https://openalex.org/W2068219766,https://doi.org/10.1093/nar/gkq164,article,Design and characterization of molecular tools...,2010,348,"Cyanobacteria are suitable for sustainable, so...","[Synthetic biology, Biology, Cyanobacteria, Co...","{'id': 'https://openalex.org/S134668137', 'tit...","[{'last_name': 'Hsin-Ho Huang'}, {'last_name':...",[Uppsala University],[Sweden],"[https://openalex.org/W2887249870, https://ope...","{'is_oa': True, 'oa_pdf': 'https://academic.ou...",False
4,https://openalex.org/W2155758567,https://doi.org/10.1111/nph.13532,article,Standards for plant synthetic biology: a commo...,2015,287,Summary Inventors in the field of mechanical a...,"[Syntax, Field (mathematics), Computer science...","{'id': 'https://openalex.org/S58631098', 'titl...","[{'last_name': 'Nicola J. Patron'}, {'last_nam...","[Joint BioEnergy Institute, University of Warw...","[United Kingdom, Germany, Netherlands , Israel...","[https://openalex.org/W2028343312, https://ope...","{'is_oa': True, 'oa_pdf': 'https://onlinelibra...",False


In [ ]:
print(pubs.title[0:10])

0     Engineering BioBrick vectors from BioBrick parts
1    A Discrete Self-Assembled Metal Array in Artif...
2    Measuring the activity of BioBrick promoters u...
3    Design and characterization of molecular tools...
4    Standards for plant synthetic biology: a commo...
5    Solution structure of a DNA double helix with ...
6    ePathBrick: A Synthetic Biology Platform for E...
7    The Bacillus BioBrick Box: generation and eval...
8    Designing and engineering evolutionary robust ...
9    Building outside of the box: iGEM and the BioB...
Name: title, dtype: object


In [ ]:
# # Identify the reviews
# reviews = pubs[pubs.title.str.contains("review|overview|landscape|survey|state-of-the-art|challenges and|and challenges|recommendations and|and recommendations", case=False)]
# print(len(reviews))
# reviews.head()

In [ ]:
# Remove the reviews
# pubs = pubs[~pubs.title.str.contains("review|overview|landscape|survey|state-of-the-art|challenges and|and challenges|recommendations and|and recommendations", case=False)].reset_index(drop=True)
# print(len(pubs))
# pubs.head()

## Save data

In [ ]:
# Save serialized version of data from Dimensions
file = open(DATAFOLDER + '/raw_dataset_pickled_PART7', 'wb')
pickle.dump(pubs, file)
file.close()



---

